In [13]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
import joblib

# Load the data
df = pd.read_csv("deliveries_with_paths.csv")

# Convert datetime columns
df['delivery_finished'] = pd.to_datetime(df['delivery_finished'])
df['delivery_started'] = pd.to_datetime(df['delivery_started'])

# Feature engineering
df['hour'] = df['delivery_finished'].dt.hour
df['minute'] = df['delivery_finished'].dt.minute
df['day_of_week'] = df['delivery_finished'].dt.dayofweek

# Calculate target (total delivery time in seconds)
df['total_delivery_time'] = (df['delivery_finished'] - df['delivery_started']).dt.total_seconds()

# Filter data for completed deliveries
df = df[df['status'] == 'COMPLETED']

# Feature selection and target variable
X = df[['hour', 'minute', 'day_of_week', 'path', 'number_of_obstacles']]
y = df['total_delivery_time']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), ['hour', 'minute', 'day_of_week', 'number_of_obstacles']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['path'])
])

# Model pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

# Hyperparameter tuning
param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2],
    'model__max_features': ['sqrt', 'log2']
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best model
best_model = grid_search.best_estimator_

# Save the trained model to a .pkl file
joblib.dump(best_model, "new_model.pkl")

print("Model trained and saved as 'new_model.pkl'.")


Model trained and saved as 'new_model.pkl'.
